# 03 — Soil QC gates and mechanism tests against literature expectations
**Fernow · plot-level (n = 6/watershed/depth)** — ArcGIS Pro Notebooks or Jupyter.

Notebook 01 established *what* differs. This notebook asks *why*, by testing directional expectations drawn from the
literature, and records each as **consistent / opposite / not detected**. Two QC gates come first because they change
what can be interpreted.

| Gate / test | Question |
|---|---|
| QC-1 | Are WS5 soil subcompartment labels consistent with the thesis and tree-plot map? |
| QC-2 | Are forest-floor (OH) **stocks** physically plausible? (concentrations are unaffected) |
| T1 | Fraction C:N — do LF/hPOM carry fresh, high-C:N inputs after harvest? MAOM stable? |
| T2 | POM share of mineral-soil C — shift toward the less-protected pool? |
| T3 | Root–LF coupling — is subsoil LF-C tracking fine-root necromass/biomass? |
| T4 | C stratification (0–20 vs 20–40) — is harvest homogenising the profile? |
| T5 | Equivalent soil mass — do fixed-depth stock comparisons survive BD differences? |
| T6 | Bulk density — compaction under harvest traffic? |
| T7 | MAOC vs mineral protective capacity (Hassink 1997) — exploratory |


In [ ]:
# ============================== CONFIG ==============================
import os, sys
NOTEBOOK_DIR = os.getcwd()
DATA_ROOT = r'.'                              # repo root (holds gdb_export/ and WV_COND_DWM_CALC.csv)
GDB = os.path.join(DATA_ROOT, 'gdb_export')
SWAP_WS5_SOIL_LABELS = True     # thesis + tree map: 5 plots WS5A, 1 plot WS5B; soil file has the inverse (QC-1)
OH_STOCKS_VALID = False         # set True only after confirming OH soil_dw_g is the whole 20x20-cm frame (QC-2)
LF_CENSOR_FILL = 'half_DL'      # 'half_DL' | 'zero' | 'full_DL' for the 3 censored WS5 20-40 cm LF-C values
# ====================================================================
sys.path.insert(0, NOTEBOOK_DIR)
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from scipy import stats
import warnings; warnings.filterwarnings('ignore')
from fernow_stats import *
pd.set_option('display.width', 160); pd.set_option('display.max_columns', 30)
soil = pd.read_csv(os.path.join(GDB, 'soil_samples.csv'))
frac = pd.read_csv(os.path.join(GDB, 'soil_fractions.csv'))
print(soil.shape, frac.shape)

## QC-1 — WS5 soil subcompartment labels
Thesis Methods: *"one plot in each managed watershed was sampled from WS2-DLC (B) … WS5-STS (B)."* The tree-plot map
agrees (WS5: 5 × WS5A, 1 × WS5B). The soil file labels WS5 as 5 × `WS5b` + 1 × `WS5a`. The subcompartment label is
not used in any test below (watershed is the factor), but it determines the time-since-harvest attached to each soil
plot, so it is corrected here when `SWAP_WS5_SOIL_LABELS = True`.


In [ ]:
lab = soil[['watershed', 'soil_plot', 'soil_subcomp']].drop_duplicates()
print('As exported:'); print(lab.groupby(['watershed', 'soil_subcomp']).size().to_string())
if SWAP_WS5_SOIL_LABELS:
    sw = {'WS5a': 'WS5b', 'WS5b': 'WS5a'}
    soil['soil_subcomp'] = soil['soil_subcomp'].replace(sw); frac['soil_subcomp'] = frac['soil_subcomp'].replace(sw)
    print('\nAfter correction:'); print(soil[['watershed', 'soil_plot', 'soil_subcomp']].drop_duplicates().groupby(['watershed', 'soil_subcomp']).size().to_string())

## QC-2 — Are organic-horizon (OH) stocks plausible?
OH was sampled in a 20 × 20 cm frame after removing fresh litter; areal mass = dry weight / 0.04 m² (the ×25 conversion
is arithmetically correct). The question is whether the recorded dry weight is the **whole frame** or a **subsample**.
Benchmark: measured duff + litter carbon on West Virginia FIA down-woody-material plots in hardwood forest types
(Woodall & Monleon 2008 protocol), converted lb ac⁻¹ → Mg ha⁻¹.


In [ ]:
oh = soil[soil.depth_code == 'OH']
print('OH dry weight per 400 cm2 frame (g):', oh.soil_dw_g.describe().round(2).to_dict())
oh_mass_Mgha = oh.wtsoil_area_gm2 / 100
oh_C_Mgha = oh.soil_C_gm2 / 100
print(f'Implied OH mass: median {oh_mass_Mgha.median():.2f} Mg/ha; OH carbon: median {oh_C_Mgha.median():.2f} Mg C/ha')

dwm_path = os.path.join(DATA_ROOT, 'WV_COND_DWM_CALC.csv'); cond_path = os.path.join(DATA_ROOT, 'WV_COND.csv')
if os.path.exists(dwm_path) and os.path.exists(cond_path):
    dwm = pd.read_csv(dwm_path, usecols=['CND_CN', 'DUFF_CARBON', 'LITTER_CARBON'])
    cc = pd.read_csv(cond_path, usecols=['CN', 'FORTYPCD', 'COND_STATUS_CD', 'CONDPROP_UNADJ'])
    hw = cc[(cc.COND_STATUS_CD == 1) & (cc.FORTYPCD.between(500, 599) | cc.FORTYPCD.between(800, 899))]
    ff = dwm.merge(hw, left_on='CND_CN', right_on='CN')
    ff_C = (ff.DUFF_CARBON.fillna(0) + ff.LITTER_CARBON.fillna(0)) * LBAC_TO_MGHA
    print(f'WV FIA hardwood duff+litter C (measured, n = {len(ff_C)}): median {ff_C.median():.1f} Mg C/ha, '
          f'10th pct {ff_C.quantile(0.10):.1f}, 90th pct {ff_C.quantile(0.90):.1f}')
    print(f'Fernow OH C median is {ff_C.median() / oh_C_Mgha.median():.0f}x below the FIA median and below the FIA '
          f'{(ff_C < oh_C_Mgha.median()).mean() * 100:.1f}th percentile.')
    print('NOTE: Fernow OH excluded fresh litter (Oi), FIA includes it — this explains a factor of ~2, not ~20-60.')
print('\nOH_STOCKS_VALID =', OH_STOCKS_VALID, '-> OH stock tests are', 'INCLUDED' if OH_STOCKS_VALID else 'EXCLUDED from inference (shown descriptively)')

**Consequence.** Unless the OH dry weights are confirmed to be whole-frame masses, every OH *stock* (g m⁻²) — including
the OH C and N Kruskal–Wallis results (p ≈ 0.02) and the forest-floor bar in the ecosystem-C figure — should not be
interpreted. OH **concentrations** (%C, %N) and **C:N** do not depend on mass and remain valid (tested in T1b).


In [ ]:
# Plot-level tables (subplots averaged first)
cols = ['soil_C_perc', 'soil_N_perc', 'soil_C_gm2', 'soil_N_gm2', 'soil_CN', 'fine_root_gm2', 'BD_gcm3', 'wtsoil_area_gm2']
sp = soil.groupby(['watershed', 'soil_plot', 'depth_code'])[cols].mean().reset_index()
W = sp.pivot_table(index=['watershed', 'soil_plot'], columns='depth_code', values=cols)
W.columns = [f'{a}|{b}' for a, b in W.columns]; W = W.reset_index()

dl = frac['LF_C_gm2'].dropna().min()
fill = {'half_DL': dl / 2, 'zero': 0.0, 'full_DL': dl}[LF_CENSOR_FILL]
frac['LF_censored'] = frac['LF_C_gm2'].isna()
frac['LF_C_f'] = frac['LF_C_gm2'].fillna(fill)
frac['POM_C'] = frac['LF_C_f'] + frac['hPOM_C_gm2']
for fr in ['LF', 'hPOM', 'MAOM']:
    frac[f'{fr}_CN'] = frac[f'{fr}_C_gm2'] / frac[f'{fr}_N_gm2']
frac['POM_share'] = frac['POM_C'] / (frac['POM_C'] + frac['MAOM_C_gm2'])
print(f'Censored LF-C values: {int(frac.LF_censored.sum())} (fill = {LF_CENSOR_FILL} = {fill:.1f} g/m2 where a C value is needed)')

results = []   # every test appends here; FDR applied at the end
def add(test, expectation, source, groups, depth=''):
    r = battery_row(test, groups, depth=depth, expectation=expectation, source=source)
    results.append(r); return r
def g3(df, col):
    return [df.loc[df.watershed == w, col].values for w in ORDER]

## T1 — Fraction C:N
**Expectations.** POM (LF, hPOM) is dominated by partially decomposed plant material with high C:N; MAOM is dominated by
microbially processed, N-rich compounds with low C:N (Cotrufo et al. 2013; Lavallee et al. 2020). *Hypothesis:* if
harvest residues and dead roots are entering the light fraction, **LF C:N rises** in recently cut watersheds; MAOM C:N,
a slow pool, should **not change** over 2–4 years. Censored LF values are excluded (no C:N without a C value).


In [ ]:
for depth in ['0-20', '20-40']:
    d = frac[frac.depth_code == depth]
    add('LF C:N', 'up (fresh residue/root inputs)', 'hypothesis; Lavallee 2020', g3(d[~d.LF_censored], 'LF_CN'), depth)
    add('hPOM C:N', 'up or ~', 'Lavallee 2020', g3(d, 'hPOM_CN'), depth)
    add('MAOM C:N', '~ (slow pool)', 'Cotrufo 2013; Lavallee 2020', g3(d, 'MAOM_CN'), depth)
print('Pooled fraction C:N (all plots) — sanity check vs. literature ordering LF > hPOM >= MAOM:')
print(frac[['LF_CN', 'hPOM_CN', 'MAOM_CN']].median().round(1).to_dict())
pd.DataFrame(results)[['test', 'depth', 'n', 'REF', 'DLC', 'STS', 'ANOVA_p', 'Welch_p', 'KW_p', 'lead_test', 'lead_p']].round(3)

### T1b — Forest-floor (OH) C concentration and C:N (valid regardless of QC-2)
**Expectation.** No consistent direction in the literature for forest-floor C:N after partial harvest; Fernow's
watersheds are N-saturated (Peterjohn et al. 1996), which argues against strong N limitation signals. Reported as
exploratory.


In [ ]:
n0 = len(results); ohp = sp[sp.depth_code == 'OH']
add('OH %C', 'down (loss of forest-floor OM)', 'Nave 2010 (stocks)', g3(ohp, 'soil_C_perc'), 'OH')
add('OH C:N', 'no directional expectation', 'Peterjohn 1996 (N-saturated site)', g3(ohp, 'soil_CN'), 'OH')
if OH_STOCKS_VALID:
    add('OH C stock', 'down ~30%', 'Nave 2010', g3(ohp, 'soil_C_gm2'), 'OH')
pd.DataFrame(results[n0:])[['test', 'depth', 'REF', 'DLC', 'STS', 'ANOVA_p', 'KW_p', 'lead_test', 'lead_p']].round(3)

## T2 — POM share of mineral-soil C
**Expectation.** Harvest adds residue and root necromass to POM before any of it can be processed into MAOM, so the
**POM share rises** shortly after cutting (Nave et al. 2010 residue incorporation; Lavallee et al. 2020 framework).
20–40 cm STS values include the censored-LF fill; the sensitivity to `LF_CENSOR_FILL` is shown.


In [ ]:
for depth in ['0-20', '20-40']:
    d = frac[frac.depth_code == depth]
    add('POM share of mineral C', 'up (new particulate inputs)', 'Nave 2010; Lavallee 2020', g3(d, 'POM_share'), depth)
d40 = frac[frac.depth_code == '20-40'].copy()
for name, val in [('zero', 0.0), ('half_DL', dl / 2), ('full_DL', dl)]:
    lf = d40['LF_C_gm2'].fillna(val); share = (lf + d40.hPOM_C_gm2) / (lf + d40.hPOM_C_gm2 + d40.MAOM_C_gm2)
    r = battery([share[d40.watershed == w].values for w in ORDER])
    print(f'20-40 POM share, censor fill = {name:8s}: means {np.round(r["means"], 3)}  ANOVA p = {r["p"]:.4f}  KW p = {r["pk"]:.4f}')

## T3 — Root–light-fraction coupling
**Hypothesis (novel test of the residue/root mechanism).** If subsoil LF-C under DLC derives from roots of harvested
trees, plots with more fine-root material should hold more LF-C. Tested pooled and **within watershed**
(group-demeaned Spearman, so the treatment difference itself cannot create the correlation). Censored LF values excluded.
Caveat: `fine_root_gm2` is live+dead roots picked from the core; it cannot separate necromass from live biomass.


In [ ]:
fr = W[['watershed', 'soil_plot', 'fine_root_gm2|0-20', 'fine_root_gm2|20-40']]
m = frac.merge(fr, on=['watershed', 'soil_plot'])
for depth in ['0-20', '20-40']:
    d = m[(m.depth_code == depth) & (~m.LF_censored)].rename(columns={f'fine_root_gm2|{depth}': 'roots'})
    rp = stats.spearmanr(d.roots, d.LF_C_gm2)
    rw, pw, dfree = within_group_corr(d, 'roots', 'LF_C_gm2', method='spearman')
    print(f'{depth}: pooled Spearman rho = {rp.correlation:+.2f} (p = {rp.pvalue:.3f}); within-watershed rho = {rw:+.2f} (p = {pw:.3f}, df = {dfree}), n = {len(d)}')
    add('Fine roots', 'down (fewer live trees)', 'mechanistic expectation', g3(W, f'fine_root_gm2|{depth}'), depth)
fig, ax = plt.subplots(figsize=(5, 4))
d = m[(m.depth_code == '20-40') & (~m.LF_censored)]
for w in ORDER:
    s = d[d.watershed == w]; ax.scatter(s['fine_root_gm2|20-40'], s.LF_C_gm2, color=COLORS[w], label=LABELS[w], s=40, edgecolor='k')
ax.set_xlabel('Fine roots 20–40 cm (g m$^{-2}$)'); ax.set_ylabel('LF-C 20–40 cm (g m$^{-2}$)'); ax.legend()
ax.set_title('Root–LF coupling, subsoil'); plt.tight_layout(); plt.show()

## T4 — Carbon stratification
Stratification ratio = %C(0–20) / %C(20–40) per plot (concentration-based, so independent of BD; Franzluebbers 2002).
**Hypothesis.** Harvest disturbance and deep residue/root inputs **lower** stratification. A stock-based ratio is also
reported.


In [ ]:
W['strat_conc'] = W['soil_C_perc|0-20'] / W['soil_C_perc|20-40']
W['strat_stock'] = W['soil_C_gm2|0-20'] / W['soil_C_gm2|20-40']
W['subsoil_share'] = W['soil_C_gm2|20-40'] / (W['soil_C_gm2|0-20'] + W['soil_C_gm2|20-40'])
add('C stratification ratio (conc.)', 'down (profile mixing / deep inputs)', 'hypothesis; Franzluebbers 2002', g3(W, 'strat_conc'), '0-20/20-40')
add('Subsoil share of 0-40 C stock', 'up', 'hypothesis', g3(W, 'subsoil_share'), '20-40 / 0-40')
pd.DataFrame(results[-2:])[['test', 'REF', 'DLC', 'STS', 'ANOVA_p', 'Welch_p', 'KW_p', 'lead_test', 'lead_p']].round(3)

## T5 — Equivalent soil mass (ESM)
Fixed-depth stocks confound C change with BD change (Ellert & Bettany 1995; Wendt & Hauser 2013; von Haden et al. 2020).
ESM compares C in the **same mineral-soil mass** per plot. Reference masses = the minimum cumulative mass across plots at
each depth (avoids extrapolation); C is interpolated linearly within layers assuming uniform concentration.


In [ ]:
M1 = W['wtsoil_area_gm2|0-20']; M2 = W['wtsoil_area_gm2|20-40']
C1 = W['soil_C_gm2|0-20']; C2 = W['soil_C_gm2|20-40']
ref20, ref40 = M1.min(), (M1 + M2).min()
def c_at(mref, M1, M2, C1, C2):
    return np.where(mref <= M1, C1 * mref / M1, C1 + C2 * (mref - M1) / M2)
W['C_ESM_20'] = c_at(ref20, M1, M2, C1, C2) / 100
W['C_ESM_40'] = c_at(ref40, M1, M2, C1, C2) / 100
W['C_FD_40'] = (C1 + C2) / 100
print(f'Reference masses: {ref20 / 1000:.0f} kg/m2 (~0-20), {ref40 / 1000:.0f} kg/m2 (~0-40)')
add('Mineral soil C, fixed depth 0-40 (Mg/ha)', 'down or ~', 'Nave 2010; James & Harrison 2016', g3(W, 'C_FD_40'), '0-40 FD')
add('Mineral soil C, ESM ~0-40 (Mg/ha)', 'down or ~', 'von Haden 2020 (method)', g3(W, 'C_ESM_40'), '0-40 ESM')
add('Mineral soil C, ESM ~0-20 (Mg/ha)', 'down or ~', 'von Haden 2020 (method)', g3(W, 'C_ESM_20'), '0-20 ESM')
pd.DataFrame(results[-3:])[['test', 'REF', 'DLC', 'STS', 'ANOVA_p', 'KW_p', 'power', 'lead_test', 'lead_p']].round(3)

## T6 — Bulk density (compaction)
**Expectation.** Heavy harvest traffic increases BD, strongest near the surface and on skid trails (Cambi et al. 2015).
Sampling was at permanent plots, not skid trails, so a weak or undetected signal is plausible.


In [ ]:
add('Bulk density', 'up (compaction)', 'Cambi 2015', g3(W, 'BD_gcm3|0-20'), '0-20')
add('Bulk density', 'up or ~', 'Cambi 2015', g3(W, 'BD_gcm3|20-40'), '20-40')
pd.DataFrame(results[-2:])[['test', 'depth', 'REF', 'DLC', 'STS', 'ANOVA_p', 'KW_p', 'lead_test', 'lead_p']].round(3)

## T7 — MAOC relative to mineral protective capacity (exploratory)
Hassink (1997): capacity (g C kg⁻¹) = 4.09 + 0.37 × (% particles <20 µm). SSURGO silt + clay is <50 µm, so capacity
is **overestimated** here and saturation ratios are lower bounds. Silt + clay is taken as the watershed mean of the
SSURGO values at the tree plots (soil plots are not individually located). See Georgiou et al. (2022) for updated
capacity estimates. Descriptive only — not included in the FDR battery.


In [ ]:
ss = pd.read_csv(os.path.join(GDB, 'ssurgo_plot.csv')).merge(pd.read_csv(os.path.join(GDB, 'plots_study.csv'))[['plot_id', 'watershed']], on='plot_id')
for depth, sfx in [('0-20', '0_20'), ('20-40', '20_40')]:
    fine = (ss[f'silttotal_r_{sfx}'] + ss[f'claytotal_r_{sfx}']).groupby(ss.watershed).mean()
    cap = 4.09 + 0.37 * fine
    d = frac[frac.depth_code == depth].merge(W[['watershed', 'soil_plot', f'wtsoil_area_gm2|{depth}']], on=['watershed', 'soil_plot'])
    d['MAOC_gkg'] = d.MAOM_C_gm2 / (d[f'wtsoil_area_gm2|{depth}'] / 1000)
    out = d.groupby('watershed').MAOC_gkg.mean().to_frame().join(cap.rename('capacity_gkg'))
    out['saturation_ratio'] = out.MAOC_gkg / out.capacity_gkg
    out.index = [LABELS[w] for w in out.index]
    print(f'\n{depth} cm'); print(out.reindex(['REF', 'DLC', 'STS']).round(2))

## Summary — literature expectations vs. observations
`lead_p` follows a pre-specified rule: Kruskal–Wallis if residuals fail Shapiro–Wilk, Welch if only Levene fails,
otherwise ANOVA (never "whichever p is smaller"). Benjamini–Hochberg FDR across all tests
in this notebook. Verdict logic: **consistent** = both managed watersheds move in the expected direction and lead
FDR-p < 0.05; **consistent (trend)** = expected direction, lead p < 0.10 uncorrected; **opposite** = significant in the
opposite direction; otherwise **not detected** (report power). Directions are relative to REF.


In [ ]:
S = pd.DataFrame(results)
S['lead_p_FDR'] = bh_fdr(S.lead_p)
def direction(r):
    d = [np.sign(r.DLC - r.REF), np.sign(r.STS - r.REF)]
    return {2: 'up', -2: 'down'}.get(int(sum(d)), 'mixed')
S['observed'] = S.apply(direction, axis=1)
def verdict(r):
    exp = r.expectation.split()[0]
    if exp in ('up', 'down'):
        if r.observed == exp and r.lead_p_FDR < 0.05: return 'consistent'
        if r.observed == exp and r.lead_p < 0.10: return 'consistent (trend)'
        if r.observed not in (exp, 'mixed') and r.lead_p < 0.05: return 'opposite'
        if r.observed not in (exp, 'mixed') and r.lead_p < 0.10: return 'opposite (trend)'
    if exp == '~':
        return 'consistent (no change)' if r.lead_p >= 0.10 else ('UNEXPECTED change' if r.lead_p < 0.05 else 'trend where none expected')
    if exp in ('up', 'down') and r.lead_p < 0.05 and r.observed == 'mixed': return 'differs, direction mixed'
    return 'not detected'
S['verdict'] = S.apply(verdict, axis=1)
S.to_csv(os.path.join(NOTEBOOK_DIR, 'soil_mechanism_tests.csv'), index=False)
S[['test', 'depth', 'expectation', 'source', 'REF', 'DLC', 'STS', 'observed', 'lead_test', 'lead_p', 'lead_p_FDR', 'power', 'verdict']].round(3)

---
### References (verify page ranges against the project citation log before submission)
Cambi et al. (2015) *For. Ecol. Manage.* 338:124–138 · Cotrufo et al. (2013) *Glob. Change Biol.* 19:988–995 ·
Ellert & Bettany (1995) *Can. J. Soil Sci.* 75:529–538 · Franzluebbers (2002) *Soil Tillage Res.* 66:95–106 ·
Georgiou et al. (2022) *Nat. Commun.* 13:3797 · Hassink (1997) *Plant Soil* 191:77–87 ·
James & Harrison (2016) *Forests* 7:308 · Lavallee et al. (2020) *Glob. Change Biol.* 26:261–273 ·
Nave et al. (2010) *For. Ecol. Manage.* 259:857–866 · Peterjohn et al. (1996) *Biogeochemistry* 35:507–522 ·
von Haden et al. (2020) *Glob. Change Biol.* 26:3759–3770 · Wendt & Hauser (2013) *Eur. J. Soil Sci.* 64:58–65 ·
Woodall & Monleon (2008) USDA FS GTR NRS-22.
